# Bengali Cyberbullying Detection — Transformer Fine-Tuning (Track 2)
## BanglaBERT / MuRIL / XLM-R | Kaggle T4 x2 + mixed precision

Fine-tunes a pretrained Bengali encoder for multi-label toxicity. Predicts the **4 toxic labels**;
`neutral` is **derived**. Split-before-anything with hard zero-leakage assertions.

### Fix for the previous "stuck after model download" hang
The previous run froze right after the weights downloaded, before the first epoch printed. Cause:
a **fast (Rust) tokenizer being called inside `__getitem__` with `DataLoader(num_workers=2)`** —
forking worker processes around a fast tokenizer deadlocks (the well-known `TOKENIZERS_PARALLELISM`
fork issue). Fixes applied here:
1. `os.environ['TOKENIZERS_PARALLELISM'] = 'false'`.
2. **Pre-tokenize** the whole split once (batched) and store tensors, so `__getitem__` only indexes
   tensors — no tokenizer call in workers (also faster).
3. `num_workers=0` for the loaders.

### High-accuracy fine-tuning recipe (fixes the earlier undertraining at 4 epochs)
- **8 epochs + early stopping (patience 3)**; **layer-wise LR decay (LLRD)**; **mean+max pooling**;
  **multi-sample dropout head**; **FGM adversarial training**; cosine schedule; AMP; `nn.DataParallel`.

Backbone `csebuetnlp/banglabert` → `google/muril-base-cased` → `xlm-roberta-base`. ~110M params —
intentionally over the 10M budget.


In [ ]:
# Section 1: Setup & Imports
!pip install -q transformers sentencepiece iterative-stratification

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'   # FIX: avoid fast-tokenizer fork deadlock

import re, math, random, time, json, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import (
    f1_score, classification_report, hamming_loss,
    roc_auc_score, average_precision_score
)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore'); sns.set_style('whitegrid')

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(SEED)

NUM_GPUS = torch.cuda.device_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device} | GPUs: {NUM_GPUS}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)')


In [ ]:
# Section 2: Configuration

class Config:
    DATA_PATH = 'combined_multi_labeled_bengali_comments_balanced_13k_14k_plus_neutral_plus_threat300.csv'
    TEXT_COL = 'text'
    LABEL_COLS = ['vulgar', 'threat', 'troll', 'insult', 'neutral']
    TOXIC_COLS = ['vulgar', 'threat', 'troll', 'insult']
    NEUTRAL_COL = 'neutral'
    NUM_OUT = 4

    MODEL_CANDIDATES = ['csebuetnlp/banglabert', 'google/muril-base-cased', 'xlm-roberta-base']

    MIN_WORDS = 2
    MAX_LEN = 128
    TEST_FRAC = 0.15
    VAL_FRAC = 0.15

    BATCH_SIZE_PER_GPU = 16
    EPOCHS = 8
    LR = 3e-5
    HEAD_LR = 1e-4
    LLRD_DECAY = 0.9
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    GRAD_CLIP = 1.0
    DROPOUT = 0.2
    NUM_DROPOUT_SAMPLES = 5
    USE_AMP = True

    USE_FOCAL_LOSS = True
    FOCAL_GAMMA = 2.0

    USE_FGM = True
    FGM_EPS = 1.0

    NUM_WORKERS = 0           # FIX: 0 avoids fast-tokenizer fork hang (data is pre-tokenized anyway)
    PATIENCE = 3
    THRESH_MIN = 0.30
    THRESH_MAX = 0.70
    THRESH_STEP = 0.02
    ENSEMBLE_SEEDS = [42, 7, 2024]

cfg = Config()
EFFECTIVE_BATCH = cfg.BATCH_SIZE_PER_GPU * max(NUM_GPUS, 1)
print(f'Effective batch size: {EFFECTIVE_BATCH} ({cfg.BATCH_SIZE_PER_GPU} x {max(NUM_GPUS,1)} GPU)')
print(f'Predicts {cfg.NUM_OUT} toxic labels; neutral derived. AMP={cfg.USE_AMP} | FGM={cfg.USE_FGM} | LLRD={cfg.LLRD_DECAY} | num_workers={cfg.NUM_WORKERS}')


In [ ]:
# Section 3: Load + light clean + dedup + derive neutral

data_paths = [
    cfg.DATA_PATH,
    f'/kaggle/input/datasets/muhammedjunayed/bengalicyber/{cfg.DATA_PATH}',
    f'/kaggle/input/bengali-cyberbullying-15k/{cfg.DATA_PATH}',
    f'./{cfg.DATA_PATH}',
]
df = None
for p in data_paths:
    if os.path.exists(p):
        df = pd.read_csv(p); print(f'Loaded: {p}'); break
if df is None:
    raise FileNotFoundError(f'Dataset not found. Tried: {data_paths}')
for col in cfg.LABEL_COLS:
    df[col] = df[col].astype(int)

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
EMOJI_RE = re.compile(
    '[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
    '\U0001F680-\U0001F6FF\U0001F900-\U0001F9FF\U00002702-\U000027B0]+', flags=re.UNICODE)

def light_clean(t):
    t = str(t)
    t = URL_RE.sub(' ', t); t = MENTION_RE.sub(' ', t); t = EMOJI_RE.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean_text'] = df[cfg.TEXT_COL].apply(light_clean)
df['_wc'] = df['clean_text'].str.split().str.len().fillna(0)
df = df[df['_wc'] >= cfg.MIN_WORDS].copy()
before = len(df)
df = df.groupby('clean_text', as_index=False)[cfg.LABEL_COLS].max()
print(f'Deduplicated on cleaned text: {before} -> {len(df)}')
tox = df[cfg.TOXIC_COLS].sum(axis=1) > 0
df[cfg.NEUTRAL_COL] = (~tox).astype(int)
df = df.reset_index(drop=True)
print('Per-class distribution:')
for c in cfg.LABEL_COLS:
    print(f'  {c:>8s}: {int(df[c].sum()):>5d} ({100*df[c].mean():.1f}%)')


## Section 4: Stratified split (before anything) with zero-leakage guarantee

In [ ]:
# Section 5: Stratified Train/Val/Test split

y = df[cfg.LABEL_COLS].values
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=cfg.TEST_FRAC, random_state=SEED)
tv_idx, test_idx = next(msss1.split(df, y))
df_tv = df.iloc[tv_idx].reset_index(drop=True)
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=cfg.VAL_FRAC/(1-cfg.TEST_FRAC), random_state=SEED)
tr_sub, val_sub = next(msss2.split(df_tv, df_tv[cfg.LABEL_COLS].values))
df_train = df_tv.iloc[tr_sub].reset_index(drop=True)
df_val   = df_tv.iloc[val_sub].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)
print(f'Train {len(df_train)} | Val {len(df_val)} | Test {len(df_test)}')

tr = set(df_train['clean_text'])
assert not (tr & set(df_val['clean_text'])) and not (tr & set(df_test['clean_text'])), 'LEAKAGE!'
for name, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    tox = d[cfg.TOXIC_COLS].sum(axis=1) > 0
    assert int(((tox & (d[cfg.NEUTRAL_COL]==1)) | (~tox & (d[cfg.NEUTRAL_COL]==0))).sum()) == 0
print('Leakage = 0 and neutral consistent in all splits.')


In [ ]:
# Section 6: Tokenizer + PRE-TOKENIZED Dataset + DataLoaders (hang fix)

tokenizer, MODEL_NAME = None, None
for name in cfg.MODEL_CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(name); MODEL_NAME = name
        print(f'Loaded tokenizer: {name}'); break
    except Exception as e:
        print(f'  tokenizer {name} failed: {type(e).__name__}')
assert tokenizer is not None, 'No tokenizer could be loaded'

class TxtDataset(Dataset):
    # Pre-tokenize the whole split ONCE (batched) and store tensors. __getitem__ then only indexes
    # tensors -> no tokenizer call inside DataLoader workers -> no fork deadlock, and faster.
    def __init__(self, df, tokenizer, cfg):
        enc = tokenizer(list(df['clean_text']), truncation=True, max_length=cfg.MAX_LEN,
                        padding='max_length', return_tensors='pt')
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels = torch.tensor(df[cfg.TOXIC_COLS].values, dtype=torch.float32)
    def __len__(self): return self.input_ids.size(0)
    def __getitem__(self, i):
        return self.input_ids[i], self.attention_mask[i], self.labels[i]

print('Pre-tokenizing splits...')
train_ds = TxtDataset(df_train, tokenizer, cfg)
val_ds   = TxtDataset(df_val, tokenizer, cfg)
test_ds  = TxtDataset(df_test, tokenizer, cfg)
train_loader = DataLoader(train_ds, batch_size=EFFECTIVE_BATCH, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=EFFECTIVE_BATCH*2, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=EFFECTIVE_BATCH*2, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
print(f'Tokenized. Train batches {len(train_loader)} | Val {len(val_loader)} | Test {len(test_loader)}')


In [ ]:
# Section 7: Model (encoder + mean&max pool + multi-sample dropout head), Focal loss, FGM

class TransformerClassifier(nn.Module):
    def __init__(self, model_name, cfg):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropouts = nn.ModuleList([nn.Dropout(cfg.DROPOUT) for _ in range(cfg.NUM_DROPOUT_SAMPLES)])
        self.classifier = nn.Linear(hidden * 2, cfg.NUM_OUT)
        nn.init.xavier_uniform_(self.classifier.weight); nn.init.zeros_(self.classifier.bias)
        self.cfg = cfg
    def forward(self, input_ids, attention_mask):
        hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (hs * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        max_pool = hs.masked_fill(mask == 0, -1e9).max(1)[0]
        pooled = torch.cat([mean_pool, max_pool], dim=-1)
        if self.training:
            return torch.stack([self.classifier(d(pooled)) for d in self.dropouts], 0).mean(0)
        return self.classifier(pooled)

class FocalBCELoss(nn.Module):
    def __init__(self, gamma=2.0): super().__init__(); self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = targets * torch.sigmoid(logits) + (1 - targets) * (1 - torch.sigmoid(logits))
        return ((1 - p_t) ** self.gamma * bce).mean()

class FGM:
    def __init__(self, model, eps=1.0, emb_keyword='word_embeddings'):
        self.model = model; self.eps = eps; self.kw = emb_keyword; self.backup = {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.kw in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    param.data.add_(self.eps * param.grad / norm)
    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup: param.data = self.backup[name]
        self.backup = {}

set_seed(SEED)
model = TransformerClassifier(MODEL_NAME, cfg)
total = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME} | total params: {total:,} ({total/1e6:.1f}M) -- intentionally > 10M')
criterion = FocalBCELoss(cfg.FOCAL_GAMMA) if cfg.USE_FOCAL_LOSS else nn.BCEWithLogitsLoss()
if NUM_GPUS > 1:
    model = nn.DataParallel(model); print(f'DataParallel across {NUM_GPUS} GPUs (T4 x2).')
model = model.to(device)
fgm = FGM(model, cfg.FGM_EPS) if cfg.USE_FGM else None


In [ ]:
# Section 8: Optimizer with Layer-wise LR Decay (LLRD) + cosine schedule + AMP

def build_llrd_param_groups(model, cfg):
    base = model.module if hasattr(model, 'module') else model
    enc = base.encoder
    no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
    seen = set(); groups = []
    head = [p for n, p in base.classifier.named_parameters()]
    for p in head: seen.add(id(p))
    groups.append({'params': head, 'lr': cfg.HEAD_LR, 'weight_decay': cfg.WEIGHT_DECAY})
    try:
        layers = enc.encoder.layer
    except AttributeError:
        layers = []
    n_layers = len(layers)
    for i, layer in enumerate(layers):
        lr_i = cfg.LR * (cfg.LLRD_DECAY ** (n_layers - 1 - i))
        decay_p = [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)]
        nodecay_p = [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)]
        for p in decay_p + nodecay_p: seen.add(id(p))
        groups.append({'params': decay_p, 'lr': lr_i, 'weight_decay': cfg.WEIGHT_DECAY})
        groups.append({'params': nodecay_p, 'lr': lr_i, 'weight_decay': 0.0})
    emb_lr = cfg.LR * (cfg.LLRD_DECAY ** (n_layers + 1))
    emb_decay = [p for n, p in enc.embeddings.named_parameters() if not any(nd in n for nd in no_decay)]
    emb_nodecay = [p for n, p in enc.embeddings.named_parameters() if any(nd in n for nd in no_decay)]
    for p in emb_decay + emb_nodecay: seen.add(id(p))
    groups.append({'params': emb_decay, 'lr': emb_lr, 'weight_decay': cfg.WEIGHT_DECAY})
    groups.append({'params': emb_nodecay, 'lr': emb_lr, 'weight_decay': 0.0})
    rest = [p for p in base.parameters() if id(p) not in seen]
    if rest:
        groups.append({'params': rest, 'lr': cfg.LR, 'weight_decay': cfg.WEIGHT_DECAY})
    return groups

optimizer = AdamW(build_llrd_param_groups(model, cfg))
total_steps = cfg.EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps*cfg.WARMUP_RATIO), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.USE_AMP and device.type == 'cuda')
print(f'Param groups: {len(optimizer.param_groups)} | Total steps {total_steps} | Warmup {int(total_steps*cfg.WARMUP_RATIO)}')


In [ ]:
# Section 9: Training loop (AMP + FGM adversarial training + early stopping)

def neutral_from_toxic(b):
    return np.concatenate([b, (b.sum(axis=1) == 0).astype(int).reshape(-1,1)], axis=1)

def amp_ctx():
    return torch.cuda.amp.autocast(enabled=cfg.USE_AMP and device.type == 'cuda')

def train_one_epoch(model, loader):
    model.train(); tot = 0.0
    for input_ids, attn, labels in loader:
        input_ids, attn, labels = input_ids.to(device), attn.to(device), labels.to(device)
        optimizer.zero_grad()
        with amp_ctx():
            loss = criterion(model(input_ids, attn), labels)
        scaler.scale(loss).backward()
        if fgm is not None:
            fgm.attack()
            with amp_ctx():
                loss_adv = criterion(model(input_ids, attn), labels)
            scaler.scale(loss_adv).backward()
            fgm.restore()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tot += loss.item()
    return tot / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); tot = 0.0; P, L = [], []
    for input_ids, attn, labels in loader:
        input_ids, attn, labels = input_ids.to(device), attn.to(device), labels.to(device)
        with amp_ctx():
            logits = model(input_ids, attn); loss = criterion(logits, labels)
        tot += loss.item(); P.append(torch.sigmoid(logits).float().cpu().numpy()); L.append(labels.cpu().numpy())
    P, L = np.vstack(P), np.vstack(L)
    yt, yp = neutral_from_toxic((L>0.5).astype(int)), neutral_from_toxic((P>0.5).astype(int))
    return tot/len(loader), f1_score(yt, yp, average='macro', zero_division=0), f1_score(yt, yp, average=None, zero_division=0), P, L

history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_per_class_f1': []}
best_f1, best_epoch, patience, best_state = 0.0, 0, 0, None
print('='*80)
for epoch in range(1, cfg.EPOCHS+1):
    t0 = time.time()
    tl = train_one_epoch(model, train_loader)
    vl, vf1, vpc, _, _ = evaluate(model, val_loader)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['val_f1'].append(vf1); history['val_per_class_f1'].append(vpc.tolist())
    if vf1 > best_f1:
        best_f1, best_epoch = vf1, epoch
        best_state = copy.deepcopy((model.module if hasattr(model,'module') else model).state_dict())
        patience = 0; mark = ' *BEST*'
    else:
        patience += 1; mark = ''
    print(f'Epoch {epoch}/{cfg.EPOCHS} | TrLoss {tl:.4f} | VaLoss {vl:.4f} | VaF1 {vf1:.4f} | {time.time()-t0:.0f}s{mark}')
    if patience >= cfg.PATIENCE:
        print(f'Early stopping at epoch {epoch}.'); break
print('='*80)
print(f'Best epoch {best_epoch} | Best Val Macro-F1 {best_f1:.4f}')
(model.module if hasattr(model,'module') else model).load_state_dict(best_state)


In [ ]:
# Section 10: Curves
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(history['train_loss'])+1)
ax[0].plot(ep, history['train_loss'], 'b-', label='Train'); ax[0].plot(ep, history['val_loss'], 'r-', label='Val')
ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(ep, history['val_f1'], 'r-o', label='Val Macro-F1')
ax[1].set_title(f'Val Macro-F1 (best {best_f1:.4f})'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle(f'Transformer ({MODEL_NAME}) - Training Curves', fontweight='bold')
plt.tight_layout(); plt.savefig('transformer_curves.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Section 11: Per-class threshold tuning on validation

_, _, _, val_preds, val_labels = evaluate(model, val_loader)

def tune_thresholds(preds, labels, cfg):
    grid = np.arange(cfg.THRESH_MIN, cfg.THRESH_MAX+1e-9, cfg.THRESH_STEP)
    best = np.full(preds.shape[1], 0.5)
    for c in range(preds.shape[1]):
        bf = -1.0
        for t in grid:
            f1 = f1_score(labels[:, c], (preds[:, c] > t).astype(int), zero_division=0)
            if f1 > bf: bf, best[c] = f1, t
    return best

tuned = tune_thresholds(val_preds, val_labels, cfg)
print('Tuned thresholds:', {c: round(float(t),2) for c,t in zip(cfg.TOXIC_COLS, tuned)})

def apply_thresholds(preds, th):
    out = np.zeros_like(preds)
    for c in range(preds.shape[1]): out[:, c] = (preds[:, c] > th[c]).astype(int)
    return out
def to5(b): return np.concatenate([b, (b.sum(axis=1)==0).astype(int).reshape(-1,1)], axis=1)


In [ ]:
# Section 12: Final test evaluation (derive neutral, report all 5 classes)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); P, L = [], []
    for input_ids, attn, y in loader:
        with amp_ctx():
            logits = model(input_ids.to(device), attn.to(device))
        P.append(torch.sigmoid(logits).float().cpu().numpy()); L.append(y.numpy())
    return np.vstack(P), np.vstack(L)

test_probs, test_toxic = predict_probs(model, test_loader)
test_pred5 = to5(apply_thresholds(test_probs, tuned))
test_true5 = to5((test_toxic > 0.5).astype(int))
test_probs5 = np.concatenate([test_probs, 1.0 - test_probs.max(axis=1, keepdims=True)], axis=1)

macro_f1 = f1_score(test_true5, test_pred5, average='macro', zero_division=0)
micro_f1 = f1_score(test_true5, test_pred5, average='micro', zero_division=0)
weighted_f1 = f1_score(test_true5, test_pred5, average='weighted', zero_division=0)
h_loss = hamming_loss(test_true5, test_pred5)
try: roc_auc = roc_auc_score(test_true5, test_probs5, average='macro')
except ValueError: roc_auc = float('nan')
try: pr_auc = average_precision_score(test_true5, test_probs5, average='macro')
except ValueError: pr_auc = float('nan')

print('='*64); print(f'   FINAL TEST EVALUATION (transformer: {MODEL_NAME})'); print('='*64)
print(f'  Macro-F1: {macro_f1:.4f} | Micro-F1: {micro_f1:.4f} | Weighted-F1: {weighted_f1:.4f}')
print(f'  Hamming: {h_loss:.4f} | ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}')
print('='*64)
print(classification_report(test_true5, test_pred5, target_names=cfg.LABEL_COLS, digits=4, zero_division=0))


In [ ]:
# Section 13: Save model + summary

base_model = model.module if hasattr(model, 'module') else model
torch.save({'model_state_dict': base_model.state_dict(), 'model_name': MODEL_NAME,
            'thresholds': tuned.tolist(), 'toxic_cols': cfg.TOXIC_COLS,
            'best_epoch': best_epoch, 'best_val_f1': best_f1,
            'test_macro_f1': macro_f1}, 'bengali_transformer_best.pt')
summary = {
    'version': 'transformer-upgraded',
    'model_name': MODEL_NAME,
    'total_params': int(sum(p.numel() for p in base_model.parameters())),
    'predicts': cfg.TOXIC_COLS, 'neutral': 'derived as NOT(any toxic)',
    'hardware': f'{NUM_GPUS}x GPU (DataParallel + AMP)',
    'tricks': ['LLRD', 'mean+max pooling', 'multi-sample dropout', 'FGM adversarial', 'cosine schedule',
               'pre-tokenized loaders (hang fix)'],
    'best_epoch': best_epoch, 'best_val_macro_f1': round(best_f1, 4),
    'test_macro_f1': round(macro_f1, 4), 'test_micro_f1': round(micro_f1, 4),
    'tuned_thresholds': {c: round(float(t),2) for c,t in zip(cfg.TOXIC_COLS, tuned)},
}
with open('transformer_summary.json', 'w') as f: json.dump(summary, f, indent=2, ensure_ascii=False)
print('Saved bengali_transformer_best.pt and transformer_summary.json')
print(f'FINAL: Test Macro-F1 = {macro_f1:.4f} | Backbone = {MODEL_NAME}')


## Section 14: Notes & tips

- **Hang fix recap:** pre-tokenized datasets + `num_workers=0` + `TOKENIZERS_PARALLELISM=false`. If
  you still see a stall, confirm Internet is enabled (for the weights download) and that a GPU is on.
- **Backbone:** `csebuetnlp/banglabert` is usually best for Bengali; for its best results normalize
  text with the authors' `normalizer`. MuRIL / XLM-R need no extra normalizer. For a higher ceiling
  try `csebuetnlp/banglabert_large` (lower batch / MAX_LEN to fit T4 memory).
- **FGM** ~doubles step time; set `USE_FGM=False` if time-constrained.
- **OOM on T4?** lower `BATCH_SIZE_PER_GPU` (8) or `MAX_LEN` (96); AMP is already on.
- **Expected:** ~0.78-0.82 with 8 epochs + LLRD + FGM (up from the 0.7654 undertrained 4-epoch run).
